In [1]:
import torch

br = torch.tensor([[0.0, 0, 0], [0, 0, 0], [0, 0, 5]])  # blob bottom-right

tl = torch.tensor([[5.0, 0, 0], [0, 0, 0], [0, 0, 0]])  # blob top-left

print("bottom-right mean:", br.mean().item())
print("top-left     mean:", tl.mean().item())


bottom-right mean: 0.5555555820465088
top-left     mean: 0.5555555820465088


In [2]:
a = torch.softmax(br.reshape(-1), dim=-1)  # flatten 3x3 -> 9, softmax over all 9
print(a.reshape(3, 3))
print("sums to:", a.sum().item())


tensor([[0.0064, 0.0064, 0.0064],
        [0.0064, 0.0064, 0.0064],
        [0.0064, 0.0064, 0.9489]])
sums to: 0.9999999403953552


In [3]:
br = torch.tensor(
    [
        [10, 10, 10],
        [10, 10, 10],
        [10, 10, 15],
    ],
    dtype=torch.float64,
)
a = torch.softmax(br.reshape(-1), dim=-1)  # flatten 3x3 -> 9, softmax over all 9
print(a.reshape(3, 3))
print("sums to:", a.sum().item())


tensor([[0.0064, 0.0064, 0.0064],
        [0.0064, 0.0064, 0.0064],
        [0.0064, 0.0064, 0.9489]], dtype=torch.float64)
sums to: 1.0


In [4]:
v, u = torch.meshgrid(torch.linspace(-1, 1, 3), torch.linspace(-1, 1, 3), indexing="ij")
print("u (column / x):")
print(u)
print("v (row / y):")
print(v)


u (column / x):
tensor([[-1.,  0.,  1.],
        [-1.,  0.,  1.],
        [-1.,  0.,  1.]])
v (row / y):
tensor([[-1., -1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  1.,  1.]])


In [5]:
x = (a.reshape(3, 3) * u).sum()
y = (a.reshape(3, 3) * v).sum()
print(f"x = {x:.4f}   y = {y:.4f}")


x = 0.9425   y = 0.9425


In [6]:
a_tl = torch.softmax(tl.reshape(-1), dim=-1).reshape(3, 3)
print(f"x = {(a_tl * u).sum():.4f}   y = {(a_tl * v).sum():.4f}")


x = -0.9425   y = -0.9425


In [7]:
for T in [5.0, 1.0, 0.2]:
    a_T = torch.softmax(br.reshape(-1) / T, dim=-1).reshape(3, 3)
    print(f"T={T:4}:  x={(a_T * u).sum():+.4f}  y={(a_T * v).sum():+.4f}")


T= 5.0:  x=+0.1603  y=+0.1603
T= 1.0:  x=+0.9425  y=+0.9425
T= 0.2:  x=+1.0000  y=+1.0000


In [8]:
two = torch.tensor([[4.0, 0, 0], [0, 0, 0], [0, 0, 5]])
for T in [1.0, 0.5, 0.2, 0.05]:
    a_T = torch.softmax(two.reshape(-1) / T, dim=-1).reshape(3, 3)
    print(f"T={T:5}:  x={(a_T * u).sum():+.4f}  y={(a_T * v).sum():+.4f}")


T=  1.0:  x=+0.4467  y=+0.4467
T=  0.5:  x=+0.7614  y=+0.7614
T=  0.2:  x=+0.9866  y=+0.9866
T= 0.05:  x=+1.0000  y=+1.0000


In [9]:
T = 1
(torch.tensor(5.0 / T).exp() - torch.tensor(4.0 / T).exp()) / (
    7 * torch.tensor(0).exp() + torch.tensor(5.0 / T).exp() + torch.tensor(4.0 / T).exp()
)

tensor(0.4467)

In [10]:
f = torch.tensor([[0.0, 0, 0], [0, 0, 0], [0, 0, 5]], requires_grad=True)
x = (torch.softmax(f.reshape(-1), -1).reshape(3, 3) * u).sum()
x.backward()
print("softmax  dx/df:\n", f.grad.round(decimals=4))

g = torch.tensor([[0.0, 0, 0], [0, 0, 0], [0, 0, 5]], requires_grad=True)
x2 = u.reshape(-1)[g.reshape(-1).argmax()]
print("\nargmax   x:", x2.item(), "  requires_grad:", x2.requires_grad)

softmax  dx/df:
 tensor([[-0.0124, -0.0060,  0.0004],
        [-0.0124, -0.0060,  0.0004],
        [-0.0124, -0.0060,  0.0546]])

argmax   x: 1.0   requires_grad: False


In [11]:
u

tensor([[-1.,  0.,  1.],
        [-1.,  0.,  1.],
        [-1.,  0.,  1.]])

In [12]:
for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
    m = torch.tensor([[0.0, 0, 0], [0, 0, 0], [0, 5 * w, 5 * (1 - w)]])
    a_m = torch.softmax(m.reshape(-1), -1).reshape(3, 3)
    print(
        f"w={w}: softmax x={(a_m * u).sum():+.4f}   argmax x={u.reshape(-1)[m.reshape(-1).argmax()]:+.1f}"
    )


w=0.0: softmax x=+0.9425   argmax x=+1.0
w=0.25: softmax x=+0.7832   argmax x=+1.0
w=0.5: softmax x=+0.3565   argmax x=+0.0
w=0.75: softmax x=+0.0470   argmax x=+0.0
w=1.0: softmax x=+0.0000   argmax x=+0.0


```py
x=tensor([ 0.9425, -0.9425, -0.9425,  0.9425], grad_fn=<MvBackward0>)
y=tensor([ 0.9425, -0.9425, -0.9425,  0.9425], grad_fn=<MvBackward0>)
torch.stack([x, y], -1)=tensor([[ 0.9425,  0.9425],
        [-0.9425, -0.9425],
        [-0.9425, -0.9425],
        [ 0.9425,  0.9425]], grad_fn=<StackBackward0>)
torch.stack([x, y], -1).reshape(B, OC * 2)=tensor([[ 0.9425,  0.9425, -0.9425, -0.9425],
        [-0.9425, -0.9425,  0.9425,  0.9425]], grad_fn=<ViewBackward0>)
```


In [13]:
from torch import nn


class SpatialSoftmax(nn.Module):
    def __init__(self, h, w):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(()))
        v, u = torch.meshgrid(torch.linspace(-1, 1, h), torch.linspace(-1, 1, w), indexing="ij")
        print(f"{u=}")
        print(f"{v=}")
        self.register_buffer("grid_u", u.reshape(-1))
        self.register_buffer("grid_v", v.reshape(-1))

    def forward(self, f):  # [B, OC, H', W']
        B, OC, H, W = f.shape
        flat = f.reshape(B * OC, H * W) / self.log_t.exp()
        a = torch.softmax(flat, dim=-1)  # [B*OC, H'*W']
        x = a @ self.grid_u  # [B*OC]
        y = a @ self.grid_v
        return torch.stack([x, y], -1).reshape(B, OC * 2)


br = torch.tensor([[0.0, 0, 0], [0, 0, 0], [0, 0, 5]])  # blob bottom-right

tl = torch.tensor([[5.0, 0, 0], [0, 0, 0], [0, 0, 0]])  # blob top-left


# 2 samples, 2 channels each. sample 0: (br, tl).  sample 1: (tl, br).
batch = torch.stack([torch.stack([br, tl]), torch.stack([tl, br])])
print("input ", batch.shape)
out = SpatialSoftmax(3, 3)(batch.float())
print("output", out.shape)
print(out.round(decimals=4))


input  torch.Size([2, 2, 3, 3])
u=tensor([[-1.,  0.,  1.],
        [-1.,  0.,  1.],
        [-1.,  0.,  1.]])
v=tensor([[-1., -1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  1.,  1.]])
output torch.Size([2, 4])
tensor([[ 0.9425,  0.9425, -0.9425, -0.9425],
        [-0.9425, -0.9425,  0.9425,  0.9425]], grad_fn=<RoundBackward1>)


In [14]:
x = torch.tensor([1, 2, 3])
y = torch.tensor([4, 5, 6])
torch.stack([x, y], -1)

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [15]:
v, u = torch.meshgrid(torch.linspace(-1, 1, 4), torch.linspace(-1, 1, 4), indexing="ij")
print("u (column / x):")
print(u)
print("v (row / y):")
print(v)


u (column / x):
tensor([[-1.0000, -0.3333,  0.3333,  1.0000],
        [-1.0000, -0.3333,  0.3333,  1.0000],
        [-1.0000, -0.3333,  0.3333,  1.0000],
        [-1.0000, -0.3333,  0.3333,  1.0000]])
v (row / y):
tensor([[-1.0000, -1.0000, -1.0000, -1.0000],
        [-0.3333, -0.3333, -0.3333, -0.3333],
        [ 0.3333,  0.3333,  0.3333,  0.3333],
        [ 1.0000,  1.0000,  1.0000,  1.0000]])


### Summary - Spatial softmax

**`u` — horizontal position.** Negative = left, positive = right.

```-
  −1.0        −0.5         0.0         +0.5        +1.0
 left edge   left-ish     centre      right-ish   right edge
```

`+0.7` = fairly right. `+0.1` = a hair right of centre. `−0.9` = nearly the left edge.

**`v` — vertical position. `−1` = TOP, `+1` = BOTTOM.** Image convention: row 0 is the top of
the picture and the index increases downward, so positive y means _lower in the image_.

#### Temperature

```-
  flat = f / T

  T small  →  divide by small number  →  values spread APART   →  PEAKED
  T large  →  divide by big number    →  values squashed       →  FLAT
```

With the blob at 5 and background at 0:

```-
  T = 0.2 :  5/0.2 = 25  vs 0   → gap 25 → blob ~100%  → x = +1.0000  (≈ argmax)
  T = 1.0 :  5/1   =  5  vs 0   → gap  5 → blob   95%  → x = +0.9425
  T = 5.0 :  5/5   =  1  vs 0   → gap  1 → blob  ~20%  → x = +0.1603  (drifts to centre)
```

**Lowering T sharpens** — the strongest cell dominates more.
**Raising T flattens** — everything looks similar and the expectation slides toward the centre.

Same intuition as temperature in an LLM sampler: low = confident and peaked, high = spread out.
